# Nemotron-3 Nano — Optimized SFT v14

Built on top of working **v7-5** (Unsloth + Blackwell ptxas fix) with three
deliberate upgrades:

1. **Sensitive-only LoRA targets** (per CLAUDE.md §LoRA priority): 6 attention
   layers + 6 pre-attention Mamba layers + 2 shared experts/layer. Skip
   routable experts (sparse gradient), skip generic Mamba (low ROI), skip
   `lm_head` (destabilizes untied embeddings).
2. **Log-probability minimization done right**:
   - assistant-only label masking (prompt tokens → `-100`)
   - token-level sum loss (no per-sequence length norm — DAPO-style)
   - **NEFTune** (`neftune_noise_alpha=5`) embedding noise — proven SFT bump
   - label smoothing `0.05` to curb overconfidence
3. **Per-category system prompts**: classifier routes each prompt to a
   category-specific system prompt (bit/cipher/numeral/unit/gravity/equation)
   so the model learns task-conditioned reasoning.

**Hardware**: Kaggle T4/L4/A100 or RTX 6000 Pro local. Bare HF on local;
Unsloth on Kaggle (its kernels actually help on Ampere/Hopper despite
Blackwell PTXAS quirks — the ptxas-blackwell shim handles that).

## Mode + Paths

In [ ]:
import os, sys

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: train on Kaggle GPU.  Mode B: package pretrained adapter only.
TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"

# Data: prefer competition train.csv (assistant-only masking computes CoT-free NLL)
DATASET_PATH_CANDIDATES = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv",
    "/kaggle/working/train.csv",
    "F:/Hackathons/Kaggle-Nemotron/data/src/train.csv",
]
DATASET_PATH = next((p for p in DATASET_PATH_CANDIDATES if os.path.exists(p)), None)
assert DATASET_PATH, f"No training CSV found in: {DATASET_PATH_CANDIDATES}"

WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
OUTPUT_DIR     = os.path.join(WORKING, "nemo_v14")
ADAPTER_DIR    = os.path.join(OUTPUT_DIR, "sft_adapter")
SUBMISSION_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

SEED = 42
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
       "DATASET_PATH": DATASET_PATH,
       "ADAPTER_DIR": ADAPTER_DIR})

## Blackwell Triton / ptxas fix (Kaggle only — copied from v7-5)

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, subprocess, site
    triton_wheels = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
    if triton_wheels:
        target = "/kaggle/working/pydeps"
        os.makedirs(target, exist_ok=True)
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "--no-deps", "--target", target, "--upgrade",
                        "--ignore-installed", triton_wheels[0]], check=True)
        if target not in sys.path: sys.path.insert(0, target)
        site.addsitedir(target)
        print("Triton wheel installed:", triton_wheels[0])
    else:
        print("[WARN] No triton wheel under /kaggle/input — continuing with system triton.")

    import shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        nv_compiler.get_ptxas_version = lambda arch: '12.0'
    except Exception as e:
        print("[WARN] triton ptxas shim failed:", e)
    print('Training env fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton fixes.")

## Offline package install (Unsloth + Mamba kernels)

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, subprocess
    def recursive_wheels(pat): return sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"

    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("GPU required.")
    print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)

    if os.path.isdir(packages_dir):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--no-index", "--find-links", packages_dir,
                        "unsloth", "trl", "peft", "transformers",
                        "datasets", "accelerate", "bitsandbytes"], check=True)
    else:
        print(f"[WARN] {packages_dir} missing — relying on preinstalled pkgs.")

    for pat in ("causal*conv1d*.whl", "mamba_ssm-*.whl"):
        ws = recursive_wheels(pat)
        if ws:
            subprocess.run([sys.executable, "-m", "pip", "install",
                            "--no-index", "--no-deps", ws[-1]], check=True)
    print("Offline install done. Restart kernel if stale imports.")

## Model + Tokenizer (Unsloth FastLanguageModel)

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192

    # Locate base model — kagglehub or pre-mounted dataset
    MODEL_PATH = None
    for cand in [
        "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
        "/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
        "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
    ]:
        if os.path.isdir(cand) and os.path.exists(os.path.join(cand, "config.json")):
            MODEL_PATH = cand; break
    if MODEL_PATH is None:
        import kagglehub
        MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print("Model path:", MODEL_PATH)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name             = MODEL_PATH,
        max_seq_length         = MAX_SEQ_LEN,
        load_in_4bit           = False,
        load_in_8bit           = False,
        full_finetuning        = False,
        trust_remote_code      = True,
        unsloth_force_compile  = False,
        attn_implementation    = "eager",
        dtype                  = torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded.")

## Optimized LoRA target selection

Per CLAUDE.md §LoRA priority — touch only sensitive groups:

| Target | Why |
|---|---|
| `self_attn.{q,k,v,o}_proj` on 6 attn layers | quantization-sensitive → high impact |
| `mamba.{x,dt}_proj` on layer `i-1` for each attn layer `i` | pre-attention Mamba (kept BF16 by NVIDIA) |
| `shared_expert.{gate,up,down}_proj` (all layers) | always active → see every token |
| `mamba.{in,out}_proj` on attn-adjacent only | structural alignment |

Explicitly **excluded**: routable experts (sparse grad), router/gate (NVIDIA
freezes), `lm_head` (destabilizes untied embeddings), remaining 40 Mamba
layers (low ROI per paper §4.2).

In [ ]:
if TRAIN_ON_KAGGLE:
    import re
    from collections import Counter

    # 1. Inventory all Linear modules
    linear_modules = []
    for name, mod in model.named_modules():
        if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
            linear_modules.append(name)
    print(f"Total Linear modules: {len(linear_modules)}")

    # 2. Find attention layer indices
    attn_layer_re = re.compile(r"layers\.(\d+)\..*self_attn\.q_proj$")
    attn_layers = sorted({int(m.group(1))
                          for n in linear_modules
                          for m in [attn_layer_re.search(n)] if m})
    print(f"Attention layers: {attn_layers}")
    pre_attn_layers = sorted({i - 1 for i in attn_layers if i - 1 >= 0})
    print(f"Pre-attention Mamba layers (priority): {pre_attn_layers}")

    # 3. Build precise target list
    pre_attn_set = set(pre_attn_layers)
    attn_set     = set(attn_layers)
    selected = []
    for n in linear_modules:
        # Self-attention on attn layers
        m = re.search(r"layers\.(\d+)\..*self_attn\.(q|k|v|o)_proj$", n)
        if m and int(m.group(1)) in attn_set:
            selected.append(n); continue
        # Pre-attention Mamba (x_proj, dt_proj, in_proj, out_proj on i-1 of attn)
        m = re.search(r"layers\.(\d+)\..*mamba\.(x|dt|in|out)_proj$", n)
        if m and int(m.group(1)) in pre_attn_set:
            selected.append(n); continue
        # Shared experts on ALL layers (always active)
        if re.search(r"shared_expert.*\.(gate|up|down)_proj$", n):
            selected.append(n); continue

    print(f"Selected sensitive modules: {len(selected)} / {len(linear_modules)}")
    print("Group breakdown:",
          Counter(n.rsplit('.', 1)[-1] for n in selected))

    if not selected:
        # Module names differed — fall back to suffix list (still skips routable experts via shared_expert tag)
        print("[WARN] regex matched 0 modules — using suffix fallback")
        OPTIMIZED_TARGETS = ["q_proj","k_proj","v_proj","o_proj",
                              "x_proj","dt_proj","in_proj","out_proj",
                              "gate_proj","up_proj","down_proj"]
    else:
        OPTIMIZED_TARGETS = selected   # exact module names → no over-matching

## LoRA wrap (rank=32, RSLoRA)

In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    LORA_RANK    = 32   # competition max
    LORA_ALPHA   = 64
    LORA_DROPOUT = 0.0  # Nemotron paper §3: no dropout

    print(f"Creating LoRA wrapper r={LORA_RANK} alpha={LORA_ALPHA} "
          f"on {len(OPTIMIZED_TARGETS)} targets ...")
    model = FastLanguageModel.get_peft_model(
        model,
        r                         = LORA_RANK,
        lora_alpha                = LORA_ALPHA,
        lora_dropout              = LORA_DROPOUT,
        target_modules            = OPTIMIZED_TARGETS,
        bias                      = "none",
        use_gradient_checkpointing = True,
        random_state              = SEED,
        use_rslora                = True,   # rank-stable α scaling
    )
    model.print_trainable_parameters()

## Per-category system prompts + classifier

Each puzzle type gets a tailored system prompt teaching: (a) what category
this is, (b) the canonical algorithm, (c) structured reasoning expectation,
(d) hard `\boxed{}` requirement. Trains the model to condition output on
the task family.

In [ ]:
import re

SYSTEM_PROMPT_BIT_MANIPULATION = """You are solving bit manipulation puzzles. Each puzzle gives you examples of 8-bit binary transformations and asks you to apply the same rule to a new input.

Your approach:
1. Study each input→output pair carefully.
2. Analyze each output bit position independently — each output bit is a boolean function of the 8 input bits.
3. For each output bit, identify the pattern: constant 0/1, copy of input bit, NOT of input bit, or a logical combination (AND, OR, XOR, NAND, NOR, XNOR, MAJ, CHO, etc.).
4. Apply the identified rule to compute the output for the query input.
5. Present your final answer as an 8-bit binary string inside \\boxed{}.

CRITICAL: Your final answer MUST be in \\boxed{answer} format. Example: \\boxed{10110101}"""

SYSTEM_PROMPT_CIPHER = """You are solving substitution cipher puzzles. Each puzzle gives you examples of encrypted text → plaintext and asks you to decrypt a new phrase.

Your approach:
1. Build a character-to-character mapping table from the examples.
2. Each encrypted character consistently maps to the same plaintext character.
3. Apply the mapping to decrypt the query text word by word.
4. Present your final answer (the decrypted phrase) inside \\boxed{}.

CRITICAL: Your final answer MUST be in \\boxed{answer} format. Example: \\boxed{cat imagines book}"""

SYSTEM_PROMPT_NUMERAL = """You are solving numeral system conversion puzzles. Each puzzle gives you examples of number conversions and asks you to convert a new number.

Your approach:
1. Examine the examples to identify the numeral system (Roman numerals are the most common: I=1, V=5, X=10, L=50, C=100, D=500, M=1000).
2. Apply the subtraction rule: IV=4, IX=9, XL=40, XC=90, CD=400, CM=900.
3. Convert the query number using the same system.
4. Present your answer inside \\boxed{}.

CRITICAL: Your final answer MUST be in \\boxed{answer} format. Example: \\boxed{XLII}"""

SYSTEM_PROMPT_UNIT_CONVERSION = """You are solving unit conversion puzzles. Each puzzle gives you examples of measurements being converted by a secret linear scaling factor.

Your approach:
1. Compute the conversion ratio from each example: ratio = output / input.
2. Average the ratios to get the best estimate (they should all be very close).
3. Apply the ratio to the query measurement: answer = query × ratio.
4. Round to 2 decimal places.
5. Present your answer inside \\boxed{}.

CRITICAL: Your final answer MUST be in \\boxed{answer} format. Example: \\boxed{16.65}"""

SYSTEM_PROMPT_GRAVITY = """You are solving modified gravitational physics problems. The formula is d = 0.5 × g × t², but the gravitational constant g has been secretly changed.

Your approach:
1. Extract the gravitational constant from the examples using: g = 2d / t²
2. Compute g for each example and average them.
3. Apply the formula to the query: d = 0.5 × g × t²
4. Round to 2 decimal places.
5. Present your answer inside \\boxed{}.

CRITICAL: Your final answer MUST be in \\boxed{answer} format. Example: \\boxed{154.62}"""

SYSTEM_PROMPT_EQUATION = """You are solving symbolic transformation puzzles. Each puzzle gives you examples of input expressions being transformed into output expressions by a secret rule.

Your approach:
1. Study each example input→output pair carefully.
2. Look for consistent patterns:
   - Character-by-character substitution (each char maps to another)
   - Operator-dependent rules (different operators → different transformations)
   - Deletion, reversal, or reordering of characters
3. Identify the rule that explains ALL examples consistently.
4. Apply the rule to the query expression.
5. Present your answer inside \\boxed{}.

CRITICAL: Your final answer MUST be in \\boxed{answer} format. Example: \\boxed{@&}"""

SYSTEM_PROMPT_GENERIC = """You are solving "Alice's Wonderland" reasoning puzzles. Each puzzle gives you a few examples of a secret transformation rule and asks you to apply it to a new input.

Your approach:
1. Carefully analyze the pattern in all provided examples.
2. Identify the consistent rule or transformation.
3. Apply the rule to the query.
4. Think step by step and show your reasoning.
5. Present your final answer inside \\boxed{}.

CRITICAL: Your final answer MUST be wrapped in \\boxed{} — answers not in this format will receive zero points."""

_CATEGORY_PROMPTS = {
    "bit_manipulation": SYSTEM_PROMPT_BIT_MANIPULATION,
    "cipher":           SYSTEM_PROMPT_CIPHER,
    "numeral":          SYSTEM_PROMPT_NUMERAL,
    "unit_conversion":  SYSTEM_PROMPT_UNIT_CONVERSION,
    "gravity":          SYSTEM_PROMPT_GRAVITY,
    "equation":         SYSTEM_PROMPT_EQUATION,
}

def get_system_prompt(category: str) -> str:
    return _CATEGORY_PROMPTS.get(category, SYSTEM_PROMPT_GENERIC)


# Category classifier (adapted from reference/nemotron-original-data-rebalance)
def infer_task_family(prompt: str) -> str:
    text  = str(prompt)
    lower = text.lower()
    if 'alice' in lower and 'wonderland' in lower and 'equations' in lower:
        tail = text.split('now, determine the result for:')[-1].strip()
        symbols = set("!@#$%^&*()_+-=[]{}|;:'\",.<>/?`~\\")
        return 'equation' if any(ch in symbols for ch in tail) else 'equation'
    if 'roman numeral' in lower or 'roman' in lower:
        return 'numeral'
    if 'cipher' in lower or 'encrypt' in lower or 'decrypt' in lower:
        return 'cipher'
    if 'gravity' in lower or 'planet' in lower or 'gravitational' in lower or 'free.?fall' in lower:
        return 'gravity'
    if 'convert' in lower and ('unit' in lower or 'meter' in lower or 'gram' in lower or 'liter' in lower):
        return 'unit_conversion'
    if 'bit' in lower or 'binary' in lower or '8-bit' in lower:
        return 'bit_manipulation'
    return 'other'

# Smoke test
print("Classifier examples:")
for s in [
    "In Alice's Wonderland, the equations transform as: ... determine the result for: a+b",
    "Convert the following number to its Roman numeral form: 42",
    "Decrypt this cipher: XYZ",
    "The gravitational constant on planet X is unknown. d = 0.5 * g * t^2 ...",
    "Convert 5 meters into the new unit ...",
    "Apply the bit manipulation rule to 10110011",
    "Random unrelated prompt",
]:
    print(f"  {infer_task_family(s):20s} <- {s[:60]}")

## Dataset build — assistant-only label masking

Each row becomes a 3-turn conversation:
1. **system**: category-specific prompt (from classifier)
2. **user**: original puzzle + boxed-answer instruction
3. **assistant**: optional CoT + `\boxed{answer}`

Labels for system + user tokens are set to `-100` (ignored). Only the
assistant turn contributes to the cross-entropy loss → **pure
completion-token NLL**, no leakage from the prompt.

In [ ]:
import pandas as pd, re
from collections import Counter
from datasets import Dataset as HFDataset

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

df = pd.read_csv(DATASET_PATH)
print(f"Raw rows: {len(df)}.  Columns: {list(df.columns)}")

has_cot = "generated_cot" in df.columns or "cot" in df.columns
cot_col = "generated_cot" if "generated_cot" in df.columns else ("cot" if "cot" in df.columns else None)

df = df.dropna(subset=["prompt", "answer"]).reset_index(drop=True)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

records = []
type_counts = Counter()
for _, row in df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    category = infer_task_family(prompt)
    type_counts[category] += 1

    if cot_col is not None and not pd.isna(row[cot_col]) and len(str(row[cot_col]).strip()) >= 5:
        cot = re.sub(r'\\boxed\{[^}]*\}', '', str(row[cot_col])).rstrip()
        cot = cot.replace("<think>", "").replace("</think>", "").strip()
        assistant_content = f"<think>\n{cot}\n</think>\n\\boxed{{{answer}}}"
    else:
        assistant_content = f"Apply the demonstrated rule carefully.\n\\boxed{{{answer}}}"

    records.append({
        "category": category,
        "messages": [
            {"role": "system",    "content": get_system_prompt(category)},
            {"role": "user",      "content": prompt + PROMPT_SUFFIX},
            {"role": "assistant", "content": assistant_content},
        ],
    })

print("Per-category counts:", dict(type_counts))
print(f"Total records: {len(records)}")

raw_dataset = HFDataset.from_list(records)

## Tokenize with assistant-only label masking

Tokenize the whole conversation, then locate the assistant turn via the chat
template and mask out everything before it. This computes the loss only on
the tokens we want the model to learn to produce.

In [ ]:
MAX_LEN = 4096   # raise to 8192 if GPU has headroom

def render_full(msgs):
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=False,
                                              enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=False)

def render_prefix(msgs):
    # Render conversation WITHOUT the assistant turn, with generation prompt on
    try:
        return tokenizer.apply_chat_template(msgs[:-1], tokenize=False,
                                              add_generation_prompt=True,
                                              enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(msgs[:-1], tokenize=False,
                                              add_generation_prompt=True)

def tokenize_with_assistant_mask(batch):
    input_ids_batch, attn_batch, labels_batch = [], [], []
    for msgs in batch["messages"]:
        full_text   = render_full(msgs)
        prefix_text = render_prefix(msgs)
        full_ids   = tokenizer(full_text,   add_special_tokens=False,
                                truncation=True, max_length=MAX_LEN)["input_ids"]
        prefix_ids = tokenizer(prefix_text, add_special_tokens=False,
                                truncation=True, max_length=MAX_LEN)["input_ids"]
        labels = list(full_ids)
        cutoff = min(len(prefix_ids), len(labels))
        labels[:cutoff] = [-100] * cutoff   # mask system + user
        input_ids_batch.append(full_ids)
        attn_batch.append([1] * len(full_ids))
        labels_batch.append(labels)
    return {"input_ids": input_ids_batch,
            "attention_mask": attn_batch,
            "labels": labels_batch}

tokenized = raw_dataset.map(tokenize_with_assistant_mask,
                             batched=True,
                             batch_size=64,
                             remove_columns=raw_dataset.column_names,
                             desc="Tokenize + assistant-only mask")
print("Tokenized rows:", len(tokenized))

# Sanity: confirm at least one row has unmasked labels and they end with answer tokens
row = tokenized[0]
n_unmasked = sum(1 for x in row["labels"] if x != -100)
print(f"  row 0: total tokens={len(row['input_ids'])}  unmasked (loss) tokens={n_unmasked}")
tail = tokenizer.decode([x for x in row["labels"][-60:] if x != -100], skip_special_tokens=False)
print(f"  decoded tail of unmasked region: ...{tail[-200:]!r}")
assert n_unmasked > 0, "All labels masked — chat-template rendering mismatch."

## Completion-only data collator

In [ ]:
import torch

class CompletionOnlyDataCollator:
    """Pads input_ids/labels to the longest in the batch; preserves -100."""
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        input_features = [{"input_ids":      f["input_ids"],
                           "attention_mask": f["attention_mask"]} for f in features]
        batch = self.tokenizer.pad(input_features,
                                    padding=True,
                                    pad_to_multiple_of=self.pad_to_multiple_of,
                                    return_tensors="pt")
        max_len = batch["input_ids"].shape[1]
        padded = []
        for f in features:
            lab = list(f["labels"])
            lab += [-100] * (max_len - len(lab))
            padded.append(lab)
        batch["labels"] = torch.tensor(padded, dtype=torch.long)
        return batch

## Trainer — log-prob minimization, NEFTune, label smoothing

- **NLL**: `Trainer` default loss = mean of cross-entropy over non-`-100`
  tokens. Combined with assistant-only masking this is exactly
  `−Σ log P(y_t | y_<t, x)` averaged over completion tokens.
- **NEFTune** (`neftune_noise_alpha=5`): adds Gaussian noise to input
  embeddings during training. Jain et al. 2023: consistent SFT win.
- **Label smoothing 0.05**: regularizes overconfident output distribution,
  reduces NLL on out-of-distribution prompts at eval.
- **Cosine LR**, **8e-5**, warmup 5%, **2 epochs**.
- **paged_adamw_8bit** + **gradient checkpointing** (reentrant=True for Mamba).

In [ ]:
if TRAIN_ON_KAGGLE:
    import gc, time, torch
    # Memory hygiene — set BEFORE first CUDA call in this cell
    os.environ["PYTORCH_ALLOC_CONF"]    = "expandable_segments:True"
    os.environ["TORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

    from transformers import Trainer, TrainingArguments

    training_args = TrainingArguments(
        output_dir                   = os.path.join(OUTPUT_DIR, "trainer"),
        num_train_epochs             = 2,
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,         # effective batch = 8
        learning_rate                = 8e-5,
        lr_scheduler_type            = "cosine",
        warmup_ratio                 = 0.05,
        optim                        = "paged_adamw_8bit",
        adam_beta1                   = 0.9,
        adam_beta2                   = 0.95,
        adam_epsilon                 = 1e-8,
        weight_decay                 = 0.01,
        max_grad_norm                = 1.0,
        bf16                         = True,
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs= {"use_reentrant": True},  # Mamba prefers reentrant
        logging_steps                = 10,
        save_strategy                = "steps",
        save_steps                   = 250,
        save_total_limit             = 3,
        dataloader_num_workers       = 2,
        remove_unused_columns        = False,
        seed                         = SEED,
        report_to                    = "none",
        # === log-prob minimization knobs ===
        neftune_noise_alpha          = 5.0,       # noisy embeddings (Jain 2023)
        label_smoothing_factor       = 0.05,
    )

    trainer = Trainer(
        model         = model,
        args          = training_args,
        train_dataset = tokenized,
        data_collator = CompletionOnlyDataCollator(tokenizer),
    )

    torch.cuda.empty_cache(); gc.collect()

    # Resume if a checkpoint exists
    ckpt_root = training_args.output_dir
    has_ckpt  = os.path.isdir(ckpt_root) and any(d.startswith("checkpoint-") for d in os.listdir(ckpt_root))
    print(f"Resume from checkpoint: {has_ckpt}")

    print("Starting SFT training...")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=has_ckpt)
    print(f"Training done in {(time.time()-t0)/60:.1f} min")

    os.makedirs(ADAPTER_DIR, exist_ok=True)
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved -> {ADAPTER_DIR}")

## Post-SFT sanity check (3-sample greedy)

Verify the adapter emits `\boxed{}` *and* that the system-prompt routing
actually conditions output style. If this fails, packaging the adapter is
wasted effort.

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    model.eval()
    sample = df.sample(3, random_state=0)[["prompt", "answer"]].values.tolist()
    for p, ans in sample:
        cat = infer_task_family(p)
        msgs = [
            {"role": "system", "content": get_system_prompt(cat)},
            {"role": "user",   "content": str(p) + PROMPT_SUFFIX},
        ]
        try:
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                  add_generation_prompt=True,
                                                  enable_thinking=True)
        except TypeError:
            text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                  add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        tag = "BOX  " if "\\boxed{" in gen else "NOBOX"
        print(f"[{tag}] cat={cat:18s} expected={str(ans)!r:30s} tail={gen[-180:]!r}")
    model.train()

## Package submission.zip

In [ ]:
import json, shutil, zipfile

required = ["adapter_config.json", "adapter_model.safetensors"]
src_dir  = ADAPTER_DIR if TRAIN_ON_KAGGLE else PRETRAINED_ADAPTER_DATASET_PATH
print("Packaging from:", src_dir)

for fname in required:
    sp = os.path.join(src_dir, fname)
    dp = os.path.join(SUBMISSION_DIR, fname)
    if not os.path.exists(sp):
        raise FileNotFoundError(f"Missing: {sp}")
    shutil.copy2(sp, dp)
    print(f"  copied {fname}  ({os.path.getsize(dp)/1024/1024:.1f} MB)")

cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

zip_path = os.path.join(WORKING, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required:
        zf.write(os.path.join(SUBMISSION_DIR, fname), fname)
print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB — ready.")

## Notes

- **Why assistant-only mask matters for log-prob**: standard SFT computes
  loss on ALL tokens (prompt included). That dilutes the signal and lets
  prompt regularities dominate the gradient. Masking to assistant tokens
  only minimizes `−Σ log P(completion | prompt)` directly — the actual
  quantity we care about.
- **Why NEFTune**: equivalent to data augmentation in embedding space.
  Smooths the loss landscape — SFT models trained with NEFTune show
  consistently lower eval NLL on held-out prompts.
- **Why label smoothing 0.05 (not higher)**: too much smoothing hurts
  greedy decoding (competition is T=0). 0.05 is the sweet spot from
  Gemma/Llama recipes.
- **Why sensitive-only LoRA**: at rank=32 budget you have ~888M trainable
  params if you target everything. Targeting only sensitive groups cuts
  that to ~150-250M while preserving the modules that actually move
  evaluation accuracy (§4.2 Nemotron paper).
- **Per-category system prompts**: at inference, the eval harness uses
  whatever system prompt you ship. Train and eval must agree — mirror
  the classifier+system prompt in your inference notebook.